# 01. 이커머스 랭킹 상품 데이터 정제 및 분석 단위 분리

## 목적

이 노트북은 실제 `이커머스 랭킹 상품 분석` 프로젝트에서 생성된 CSV 파일을 기반으로, 크롤링·통합 직후 상품 데이터를 분석 가능한 형태로 정제하고 검증한 과정을 정리한다.

이 노트북은 임의 샘플 데이터가 아니라 실제 프로젝트 산출물을 사용한다.

사용 파일:

- `raw_integrated_ecommerce_ranking.csv`
- `cleaned_ecommerce_ranking.csv`
- `ranking_view_ecommerce.csv`
- `unique_product_ecommerce.csv`
- `preprocessing_step2_quality_check.csv`
- `preprocessing_step2_summary.csv`

## 보여주는 역량

- 크롤링/통합 직후 원본 데이터 구조 파악
- 가격, 원래가격, 할인율, 리뷰 수, 평점의 숫자형 변환 검증
- `product_key` 기반 상품명 정규화 및 중복 완화 결과 확인
- 원본 랭킹 기준 데이터와 고유 상품 기준 데이터 분리
- 랭킹 기준별 행 수, 순위 범위, 결측, 중복 여부 검증
- 전처리 결과를 분석 가능한 산출물 CSV로 정리

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/01_product")
OUTPUT_DIR = Path("../outputs/01_product")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(DATA_DIR / "raw_integrated_ecommerce_ranking.csv")
cleaned = pd.read_csv(DATA_DIR / "cleaned_ecommerce_ranking.csv")
ranking_view = pd.read_csv(DATA_DIR / "ranking_view_ecommerce.csv")
unique_product = pd.read_csv(DATA_DIR / "unique_product_ecommerce.csv")
quality_check = pd.read_csv(DATA_DIR / "preprocessing_step2_quality_check.csv")
step2_summary = pd.read_csv(DATA_DIR / "preprocessing_step2_summary.csv")

print("raw:", raw.shape)
print("cleaned:", cleaned.shape)
print("ranking_view:", ranking_view.shape)
print("unique_product:", unique_product.shape)
print("quality_check:", quality_check.shape)
print("step2_summary:", step2_summary.shape)

## 1. 데이터셋 스키마 확인

분석 전 각 CSV의 행 수, 컬럼 수, 결측치 수, 중복 행 수를 확인한다.  
이 단계는 전처리 산출물이 어떤 역할을 하는지 구분하기 위한 기본 점검이다.

In [ ]:
def schema_row(name, df):
    return {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "columns_list": ", ".join(df.columns.astype(str).tolist())
    }

schema_summary = pd.DataFrame([
    schema_row("raw_integrated_ecommerce_ranking", raw),
    schema_row("cleaned_ecommerce_ranking", cleaned),
    schema_row("ranking_view_ecommerce", ranking_view),
    schema_row("unique_product_ecommerce", unique_product),
    schema_row("preprocessing_step2_quality_check", quality_check),
    schema_row("preprocessing_step2_summary", step2_summary),
])

schema_summary.to_csv(OUTPUT_DIR / "01_dataset_schema_summary.csv", index=False, encoding="utf-8-sig")
schema_summary

## 2. 원본 데이터의 문자열 숫자형 변환 재현

`raw_integrated_ecommerce_ranking.csv`에는 가격, 원래가격, 할인율 등이 문자열 형태로 남아 있다.  
분석을 위해서는 쉼표, 원, %, 없음 등의 표현을 제거하고 숫자형으로 변환해야 한다.

여기서는 원본 데이터에서 숫자형 변환을 재현해 전처리 로직을 확인한다.

In [ ]:
def to_number(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text in ["", "없음", "nan", "None"]:
        return np.nan
    cleaned_value = re.sub(r"[^0-9.]", "", text)
    if cleaned_value == "":
        return np.nan
    return float(cleaned_value)

raw_reprocessed = raw.copy()
raw_reprocessed["price_num_recalc"] = raw_reprocessed["price"].apply(to_number)
raw_reprocessed["original_price_num_recalc"] = raw_reprocessed["original_price"].apply(to_number)
raw_reprocessed["discount_rate_num_recalc"] = raw_reprocessed["discount_rate"].apply(to_number)
raw_reprocessed["review_count_num_recalc"] = raw_reprocessed["review_count"].apply(to_number)
raw_reprocessed["rating_num_recalc"] = pd.to_numeric(raw_reprocessed["rating"], errors="coerce")

raw_reprocessed[[
    "platform", "ranking_type", "rank", "product_name",
    "price", "price_num_recalc",
    "original_price", "original_price_num_recalc",
    "discount_rate", "discount_rate_num_recalc",
    "review_count", "review_count_num_recalc",
    "rating", "rating_num_recalc"
]].head()

## 3. 전처리 전후 행 수와 상품 키 변화 확인

원본 통합 데이터, 기본 전처리 데이터, 랭킹 기준 분석 데이터, 고유 상품 기준 데이터의 행 수와 상품 키 수를 비교한다.

이 프로젝트에서는 분석 목적에 따라 두 가지 기준 데이터를 분리했다.

- `ranking_view_ecommerce.csv`: 랭킹 화면 기준 분석용 데이터
- `unique_product_ecommerce.csv`: 고유 상품 기준 분석용 데이터

In [ ]:
raw_cleaned_comparison = pd.DataFrame([
    {
        "check_item": "raw_rows",
        "value": len(raw),
        "description": "통합 직후 원본 행 수"
    },
    {
        "check_item": "cleaned_rows",
        "value": len(cleaned),
        "description": "기본 전처리 후 행 수"
    },
    {
        "check_item": "ranking_view_rows",
        "value": len(ranking_view),
        "description": "랭킹 화면 기준 분석 데이터 행 수"
    },
    {
        "check_item": "unique_product_rows",
        "value": len(unique_product),
        "description": "고유 상품 기준 데이터 행 수"
    },
    {
        "check_item": "raw_unique_product_key_raw_check",
        "value": raw["product_key_raw_check"].nunique(),
        "description": "원본 통합 단계의 상품 키 후보 고유 수"
    },
    {
        "check_item": "cleaned_unique_product_key",
        "value": cleaned["product_key"].nunique(),
        "description": "전처리 후 product_key 고유 수"
    },
    {
        "check_item": "unique_product_key_count",
        "value": unique_product["product_key"].nunique(),
        "description": "고유 상품 테이블의 product_key 고유 수"
    },
])

raw_cleaned_comparison.to_csv(OUTPUT_DIR / "01_raw_cleaned_comparison.csv", index=False, encoding="utf-8-sig")
raw_cleaned_comparison

## 4. 랭킹 기준별 전처리 품질 점검

각 플랫폼/랭킹 기준별로 100개씩 수집되었는지, 순위 범위가 1~100인지, 가격/리뷰/평점 결측이나 중복 키가 있는지 확인한다.

이 단계는 수집 데이터가 분석 가능한 랭킹 단위로 안정적으로 구성되었는지 검증하기 위한 과정이다.

In [ ]:
ranking_type_summary = (
    cleaned.groupby(["platform", "ranking_type", "ranking_label"], dropna=False)
    .agg(
        rows=("product_name", "count"),
        unique_product_key=("product_key", "nunique"),
        rank_min=("rank", "min"),
        rank_max=("rank", "max"),
        missing_price=("price", lambda x: x.isna().sum()),
        missing_review_count=("review_count", lambda x: x.isna().sum()),
        missing_rating=("rating", lambda x: x.isna().sum()),
        duplicate_key_rows=("product_key", lambda x: x.duplicated().sum()),
        top20_count=("is_top20", "sum"),
    )
    .reset_index()
)

ranking_type_summary.to_csv(OUTPUT_DIR / "01_ranking_type_summary.csv", index=False, encoding="utf-8-sig")
ranking_type_summary

기존 전처리 단계에서 생성한 품질 점검 파일도 함께 확인한다.

In [ ]:
quality_check

## 5. 플랫폼별 전처리 요약

전처리된 데이터 기준으로 플랫폼별 가격, 리뷰 수, 평점, 중복 여부를 요약한다.

이 요약은 이후 EDA에서 플랫폼별 상품 경쟁 구조를 비교하는 기반이 된다.

In [ ]:
platform_summary = (
    cleaned.groupby("platform", dropna=False)
    .agg(
        rows=("product_name", "count"),
        unique_product_key=("product_key", "nunique"),
        median_price=("price", "median"),
        mean_price=("price", "mean"),
        median_review_count=("review_count", "median"),
        mean_review_count=("review_count", "mean"),
        median_rating=("rating", "median"),
        mean_rating=("rating", "mean"),
        duplicate_overall_count=("duplicate_overall", "sum"),
    )
    .reset_index()
)

for col in ["mean_price", "mean_review_count", "mean_rating"]:
    platform_summary[col] = platform_summary[col].round(2)

platform_summary.to_csv(OUTPUT_DIR / "01_platform_preprocessing_summary.csv", index=False, encoding="utf-8-sig")
platform_summary

기존 전처리 요약 파일도 함께 확인한다.

In [ ]:
step2_summary

## 6. 랭킹 기준 데이터와 고유 상품 기준 데이터 분리 검증

이 프로젝트에서는 같은 상품이 여러 랭킹 기준에 등장할 수 있으므로, 분석 목적에 따라 데이터셋을 분리했다.

- 랭킹 화면 분석: `ranking_view_ecommerce.csv`
- 고유 상품 분석: `unique_product_ecommerce.csv`

이 분리는 랭킹 화면의 노출 구조와 고유 상품 구성 분석을 혼동하지 않기 위한 전처리 의사결정이다.

In [ ]:
duplicate_summary = pd.DataFrame([
    {
        "metric": "ranking_view_rows",
        "value": len(ranking_view),
        "description": "랭킹 기준 원본 분석용 행 수"
    },
    {
        "metric": "unique_product_rows",
        "value": len(unique_product),
        "description": "고유 상품 기준 행 수"
    },
    {
        "metric": "coupang_unique_products",
        "value": int((unique_product["platform"] == "coupang").sum()),
        "description": "쿠팡 고유 상품 수"
    },
    {
        "metric": "naver_unique_products",
        "value": int((unique_product["platform"] == "naver").sum()),
        "description": "네이버쇼핑 고유 상품 수"
    },
    {
        "metric": "multiple_ranking_type_products",
        "value": int(unique_product["is_in_multiple_ranking_types"].sum()),
        "description": "여러 랭킹 기준에 중복 등장한 고유 상품 수"
    },
])

duplicate_summary.to_csv(OUTPUT_DIR / "01_duplicate_unique_product_summary.csv", index=False, encoding="utf-8-sig")
duplicate_summary

## 7. 공개용 분석 샘플 저장

GitHub에서 빠르게 확인할 수 있도록, 핵심 컬럼만 남긴 공개용 전처리 샘플을 저장한다.

In [ ]:
public_cols = [
    "platform", "category", "ranking_type", "ranking_label", "rank",
    "product_name", "product_key", "price", "original_price", "discount_rate",
    "discount_amount", "delivery_type", "review_count", "rating",
    "price_group", "review_group", "is_top20",
    "duplicate_in_ranking_type", "duplicate_in_platform", "duplicate_overall",
    "collected_date", "source_file"
]

cleaned_public = cleaned[[col for col in public_cols if col in cleaned.columns]].copy()
cleaned_public.to_csv(OUTPUT_DIR / "01_cleaned_product_analysis_sample.csv", index=False, encoding="utf-8-sig")
cleaned_public.head()

## 포트폴리오 포인트

이 노트북은 크롤링 상품 데이터를 단순히 수집한 뒤 바로 분석한 것이 아니라, 분석 목적에 맞게 데이터 기준을 분리하고 검증했음을 보여준다.

핵심 전처리 판단은 다음과 같다.

1. 가격, 원래가격, 할인율, 리뷰 수, 평점을 숫자형으로 변환했다.
2. 상품명 기반 `product_key`를 사용해 중복 후보를 관리했다.
3. 랭킹 화면 기준 데이터와 고유 상품 기준 데이터를 분리했다.
4. 플랫폼/랭킹 기준별 행 수, 순위 범위, 결측, 중복 여부를 검증했다.

이 과정을 통해 이후 가격 분포, 리뷰 수 분포, 랭킹 겹침, 키워드 분석이 가능한 데이터셋을 구성했다.